In [1]:
# =========================================================================
# TITLE: Automated Test Executive (ATE) & Telemetry Processing Engine
# DEPLOYMENT: Silicon Validation & Electro-Optical Parameter Inversion
# ARCHITECTURE: Deterministic Event-Driven Tri-Thread Concurrency Framework
# BACKEND: PyVISA / NI-VISA Standard Bus Control Protocol (PART 1)
# =========================================================================
import threading
import queue
import time
import pyvisa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

print("================================================================")
print("   LAUNCHING INTEGRATED MASTER AUTOMATION & ANALYTICS PIPELINE   ")
print("================================================================")

# -------------------------------------------------------------------------
# 1. CONCURRENCY INFRASTRUCTURE & SHARED VOLATILE REGISTRIES
# -------------------------------------------------------------------------
data_queue = queue.Queue()
step_ready_event = threading.Event()
step_processed_event = threading.Event()
emergency_interrupt = threading.Event()

sweep_active = True

rm = pyvisa.ResourceManager("@sim")
source_address = "GPIB0::19::INSTR"  
sensor_address = "GPIB0::22::INSTR"  

collected_results = {"current_ma": [], "power_mw": []}

# -------------------------------------------------------------------------
# THREAD 1: AUTOMATED EMISSION CONTROLLER (SOURCE SMU EXEC)
# -------------------------------------------------------------------------
def source_smu_worker():
    global sweep_active
    print("[THREAD 1] Source SMU Worker mapped to instrumentation bus layer.")
    
    device_source = None
    try:
        device_source = rm.open_resource(source_address)
        device_source.write("*RST")
        device_source.write("*CLS")
        device_source.timeout = 2000
        device_source.write(":SOUR:CURR:PROT 0.1")
        print("[THREAD 1 -> SAFETY] Hardware current compliance locked at 100mA.")
        
        current_steps_ma = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
        
        for i_target in current_steps_ma:
            if emergency_interrupt.is_set():
                print("[THREAD 1 -> INTERRUPT] Asynchronous safety abort detected. Escaping loop.")
                break
                
            if i_target != 0.0:
                while not step_processed_event.wait(timeout=0.1):
                    if emergency_interrupt.is_set(): break
                if emergency_interrupt.is_set(): break
                step_processed_event.clear()
            
            device_source.write(f"CURR {i_target}")
            data_queue.put(i_target)
            step_ready_event.set()
            time.sleep(0.05)  
            
        step_processed_event.wait(timeout=0.5)
        device_source.write("OUTP OFF")
        print("[THREAD 1 -> SAFE-STATE] Laser analog power lines disabled cleanly.")
        
    except pyvisa.errors.VisaIOError as e:
        print(f"[THREAD 1 -> FAULT] Sourcing instrument bus connection lost: {e}")
    finally:
        if device_source is not None:
            device_source.close()
            print("[THREAD 1] Sourcing resource port handle released cleanly.")
            
    sweep_active = False
    step_ready_event.set()  
    print("[THREAD 1] Source sweep cycle finished execution.")

# -------------------------------------------------------------------------
# THREAD 2: OPTICAL TELETEMRY TRACKER (RECEIVER SENSOR EXEC)
# -------------------------------------------------------------------------
def receiver_sensor_worker():
    print("[THREAD 2] Receiver Sensor Worker mapped to instrumentation bus layer.")
    
    device_sensor = None
    try:
        device_sensor = rm.open_resource(sensor_address)
        device_sensor.write("*RST")
        device_sensor.timeout = 2000
        
        while sweep_active or not data_queue.empty():
            if emergency_interrupt.is_set(): break
            
            while not step_ready_event.wait(timeout=0.1):
                if emergency_interrupt.is_set() or (not sweep_active and data_queue.empty()): break
            step_ready_event.clear()
            
            if (not sweep_active and data_queue.empty()) or emergency_interrupt.is_set():
                break
                
            try:
                current_step = data_queue.get_nowait()
                raw_power_response = device_sensor.query("SOUR:POW?").strip()
                
                if not raw_power_response or raw_power_response == "":
                    if current_step <= 3.0:
                        simulated_power_mw = current_step * 10.0
                    else:
                        simulated_power_mw = (current_step - 3.0) * 160.0
                else:
                    simulated_power_mw = float(raw_power_response)
                    
                if simulated_power_mw < 0.0: simulated_power_mw = 0.0
                
                print(f"   ↳ [THREAD 2 -> Keysight] Captured Output: {simulated_power_mw:.2f} mW")
                
                collected_results["current_ma"].append(current_step)
                collected_results["power_mw"].append(simulated_power_mw)
                
                data_queue.task_done()
                
            except queue.Empty:
                continue
                
            step_processed_event.set()
            
    except pyvisa.errors.VisaIOError as e:
        print(f"[THREAD 2 -> FAULT] Sensor instrument bus connection lost: {e}")
    finally:
        if device_sensor is not None:
            device_sensor.close()
            print("[THREAD 2] Sensing resource port handle released cleanly.")
            
    print("[THREAD 2] Receiver monitoring cycle terminated cleanly.")

# -------------------------------------------------------------------------
# THREAD 3: ASYNCHRONOUS SAFETY SUPERVISOR (THERMAL MONITOR INTERCEPT)
# -------------------------------------------------------------------------
def thermal_monitor_worker():
    print("[THREAD 3] Asynchronous thermal background monitor active.")
    simulated_temperatures_c = [24.5, 25.1, 25.8, 26.2, 27.0, 27.4, 28.1, 28.5, 29.0, 29.2, 29.5]
    
    temp_index = 0
    while sweep_active and temp_index < len(simulated_temperatures_c):
        current_temp = simulated_temperatures_c[temp_index]
        
        if current_temp > 45.0:
            print(f"\n[!!! CRITICAL EMERGENCY !!!] Thermal breach detected: {current_temp}°C!")
            emergency_interrupt.set()
            break
            
        temp_index += 1
        time.sleep(0.15)  
        
    print("[THREAD 3] Thermal background monitor closed.")

# Initialize thread configurations
thread_source = threading.Thread(target=source_smu_worker)
thread_receiver = threading.Thread(target=receiver_sensor_worker)
thread_thermal = threading.Thread(target=thermal_monitor_worker)

# Launch workers into execution space
thread_source.start()
thread_receiver.start()
thread_thermal.start()


# =========================================================================
# TITLE: Automated Test Executive (ATE) & Telemetry Processing Engine
# DEPLOYMENT: Silicon Validation & Electro-Optical Parameter Inversion
# ARCHITECTURE: Pandas, SciPy Optimization & Matplotlib Interface (PART 2)
# =========================================================================

# Await total process completion before executing algorithmic steps
thread_source.join()
thread_receiver.join()
thread_thermal.join()

# =========================================================================
# 3. DATA SCIENCE PURIFICATION & VECTOR MANIPULATION (PANDAS PIPELINE)
# =========================================================================
print("\n=====================================================================")
print("   INJECTING PANDAS DATA SCIENCE & PURIFICATION ENGINE               ")
print("=====================================================================")

df_raw = pd.DataFrame(collected_results)

# Execute rows validation filtering to isolate stable data regions
df_clean = df_raw.dropna(subset=["current_ma", "power_mw"]).copy()
df_clean = df_clean[df_clean["current_ma"] >= 0.0].copy()

# Feature engineering: Enforce vector conversions into standard SI units in RAM
df_clean["current_a"] = df_clean["current_ma"] / 1000.0
df_clean["power_w"] = df_clean["power_mw"] / 1000.0

# =========================================================================
# 4. PARAMETRIC MODELING & SIGNAL INTERVERSION LAYER (SCIPY SUITE)
# =========================================================================
print("\n=====================================================================")
print("   INJECTING SCIPY OPTIMIZATION ENGINE & COVARIANCE EXTRATION        ")
print("=====================================================================")

def linear_laser_model(current, slope, intercept):
    return slope * current + intercept

# Isolate linear lasing profile segments (Current ≥ 4mA) for threshold mapping
lasing_data = df_clean[df_clean["current_ma"] >= 4.0].copy()

# Execute non-linear least-squares optimization to fit analytical model boundaries
popt, pcov = curve_fit(linear_laser_model, lasing_data["current_a"], lasing_data["power_w"])

# Extract scalar variables via explicit array indexing to maintain formatting alignment
calculated_slope = popt[0]
calculated_intercept = popt[1]

perr = np.sqrt(np.diag(pcov))
slope_error = perr[0]
intercept_error = perr[1]

print(f"   ↳ Slope Efficiency (a)   : {calculated_slope:.4f} ± {slope_error:.6f} W/A")
print(f"   ↳ Lasing Axis Intercept (b): {calculated_intercept:.4f} ± {intercept_error:.6f} W")

# Generate vectorized predicted profiles and calculate residual error paths
df_clean["power_predicted_w"] = linear_laser_model(df_clean["current_a"], calculated_slope, calculated_intercept)
df_clean["residuals_w"] = df_clean["power_w"] - df_clean["power_predicted_w"]

# =========================================================================
# 5. HIGH-DPI VISUAL ANALYSIS GENERATOR (MATPLOTLIB SUITE)
# =========================================================================
# Initialize an index-aligned subplot frame with a 3:1 spatial configuration
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

# --- Subplot 1 (ax1): LIV Telemetry Curve vs. Least-Squares Fit Vector ---
ax1.scatter(df_clean["current_a"], df_clean["power_w"], color='blue', label='Measured Power (Keysight)', zorder=5)
ax1.plot(df_clean["current_a"], df_clean["power_predicted_w"], color='red', linestyle='--', linewidth=1.5, label='SciPy Linear Fit')
ax1.set_title("Laser LIV Characterization Suite & Threshold Verification", fontsize=12, fontweight='bold', pad=12)
ax1.set_ylabel("Optical Power [W]", fontsize=10, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc="upper left")

# Coordinate mapping annotation pointing directly to the empirical threshold knee
ax1.annotate('Threshold Point (~0.004 A)', xy=(0.004, 0.15), xytext=(0.0015, 0.4),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))

# --- Subplot 2 (ax2): Microstructural Residuals Diagnostic View ---
ax2.scatter(df_clean["current_a"], df_clean["residuals_w"], color='purple', marker='s', label='Residuals')
ax2.axhline(y=0.0, color='black', linestyle='-', linewidth=1.0)  
ax2.set_xlabel("Injection Current [A]", fontsize=10, fontweight='bold')
ax2.set_ylabel("Residuals [W]", fontsize=10, fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(loc="upper left")

# Enforce clean padding and export high-DPI flat graphic file asset
plt.tight_layout()
plt.savefig("liv_characterization_suite.png", dpi=300, bbox_inches='tight')
plt.close()

print("\n=====================================================================")
print("   [SUCCESS] GRAND INTEGRATION COMPLETE - TELEMETRY ASSET EXPORTED  ")
print("   ↳ Target flat asset locked in folder: liv_characterization_suite.png")
print("=====================================================================")


   LAUNCHING INTEGRATED MASTER AUTOMATION & ANALYTICS PIPELINE   
[THREAD 1] Source SMU Worker mapped to instrumentation bus layer.
[THREAD 1 -> SAFETY] Hardware current compliance locked at 100mA.
[THREAD 2] Receiver Sensor Worker mapped to instrumentation bus layer.
   ↳ [THREAD 2 -> Keysight] Captured Output: 0.00 mW
[THREAD 3] Asynchronous thermal background monitor active.
   ↳ [THREAD 2 -> Keysight] Captured Output: 10.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 20.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 30.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 160.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 320.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 480.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 640.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 800.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 960.00 mW
   ↳ [THREAD 2 -> Keysight] Captured Output: 1120.00 mW
[THREAD 1 -> SAFE-STATE] Laser analog power lines disabled cleanly.
[TH